# EMSE scale-up calibration (19-event mutation cohort, `data_scaleup/`)

Settings: **Accelerator = CPU only**, **Internet = ON** (for `pip install`). Add the dataset
`kaggle_bundle_scaleup.zip` (uploaded separately as a Kaggle Dataset) as this notebook's input
before running. Do the whole run in **one session** (Kaggle can swap the underlying CPU across sessions,
which would invalidate cost comparisons within the run).

This is the SAME procedure as the original `kaggle_calib.ipynb` used for the primary frozen corpus, but
pointed at the parallel `data_scaleup/` root and the 19-event scale-up ledger
(`data_scaleup/events/events.json`: the original 9 mutation operators plus 10 new pre-registered ones
from Phase A). It produces `data_scaleup/profile_baseline/profiles.json`,
`data_scaleup/derived/labels.json`, `data_scaleup/manifest_static/manifest_static.json`, and zips the
write-once `data_scaleup/raw/<event_id>/<test_id>/result.json` artifacts for gate 6
(`label_reproducibility`) to be independently re-verified back in the paper repo.

**This notebook never touches `data/` (the primary frozen corpus).** Every path below is under
`data_scaleup/`.

In [ ]:
# Cell 1 -- pinned, quiet environment
# cart/cli.py imports PyYAML directly, and it is NOT a Qiskit dependency, so install it explicitly.
import os
for k, v in {"RAYON_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "QISKIT_IN_PARALLEL": "TRUE"}.items():
    os.environ[k] = v
!pip -q install qiskit==2.4.2 pyyaml numpy
import platform, subprocess
print("python", platform.python_version())
print(subprocess.run(["bash", "-lc", "grep -m1 'model name' /proc/cpuinfo; nproc"],
                      capture_output=True, text=True).stdout)  # <-- RECORD THIS for the paper's reproducibility statement

In [ ]:
# Cell 2 -- stage the project from the attached dataset, robustly.
# Kaggle sometimes auto-extracts an uploaded .zip dataset, sometimes it leaves it as a .zip file inside
# the dataset folder. Handle both, then verify src/cart AND data_scaleup/events/events.json are really
# there before moving on.
import glob, os, shutil, zipfile

!mkdir -p /kaggle/working/proj

proj_root = None

hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
if hits:
    proj_root = os.path.dirname(hits[0])
    print("found extracted project at:", proj_root)
else:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    print("no extracted pyproject.toml found; zip candidates:", zips)
    assert zips, "No dataset found under /kaggle/input at all -- did you attach kaggle_bundle_scaleup.zip as this notebook's input?"
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall("/kaggle/working/proj")
    proj_root = "/kaggle/working/proj"
    print("unzipped", zips[0], "into", proj_root)

if proj_root != "/kaggle/working/proj":
    !cp -r {proj_root}/* /kaggle/working/proj/

%cd /kaggle/working/proj
!ls -la
assert os.path.isdir("src/cart"), "src/cart still missing after staging -- inspect the ls -la output above"
assert os.path.exists("data_scaleup/events/events.json"), \
    "data_scaleup/events/events.json missing -- the bundle must include the Phase A ledger, not just src/"
print("OK: src/cart and data_scaleup/events/events.json are present")
import sys; sys.path.insert(0, "src")

In [ ]:
# Cell 3 -- sanity check: 19 scale-up events, none overlapping the primary 9 mut- ids
import json
ev = json.loads(open("data_scaleup/events/events.json").read())
ids = sorted(e["event_id"] for e in ev["events"])
print(f"{len(ids)} events in data_scaleup/events/events.json")
for i in ids:
    print(" ", i)
assert len(ids) == 19, f"expected 19, got {len(ids)}"
assert all(i.startswith("mut2-") for i in ids), "unexpected event_id prefix -- wrong ledger?"

In [ ]:
# Cell 4 -- calibration: fresh raw, 116-unit costhet corpus, 5-run warmup-discarded median
# out-data must be inside data_scaleup/, NOT the default data/manifest_static -- units_for() reads
# whatever manifest_static/manifest_static.json sits under the data_dir you pass to ground-truth/
# evaluate later, so this has to match --out-root below exactly.
!rm -rf data_scaleup/raw && mkdir -p data_scaleup/raw data_scaleup/derived data_scaleup/profile_baseline data_scaleup/manifest_static
!PYTHONPATH=src python -m cart.cli manifest --profile costhet --out-data data_scaleup/manifest_static --no-smoke
!PYTHONPATH=src python -m cart.cli ground-truth --out-root data_scaleup --events data_scaleup/events/events.json --unit-limit 116 --cost-repeats 5
import os
assert os.path.exists("data_scaleup/profile_baseline/profiles.json"), \
    "profiles.json was not produced -- scroll up in this cell's output for the actual error from manifest/ground-truth"

In [ ]:
# Cell 5 -- surface the deliverables, INCLUDING the raw write-once artifacts
import shutil, os
shutil.make_archive("data_scaleup_raw", "zip", "data_scaleup/raw")
for f in ["data_scaleup/profile_baseline/profiles.json",
          "data_scaleup/derived/labels.json",
          "data_scaleup/manifest_static/manifest_static.json",
          "data_scaleup_raw.zip"]:
    print(f, os.path.getsize(f), "bytes")

In [ ]:
# Cell 6 (sanity check, still on Kaggle) -- gates should be 6/6 right here since data_scaleup/raw/ is fresh.
# Uses SELECTORS_SCALEUP so the new `decomposed` selector (Phase A) is validated too, not just the
# original 6 baselines/proposed.
!PYTHONPATH=src python -c "
from cart.gates.validity import run_all_gates
from cart.selectors.registry_scaleup import SELECTORS_SCALEUP
import json
print(json.dumps(run_all_gates('data_scaleup', selectors=SELECTORS_SCALEUP), indent=2))
"

## After this run

Download from the Kaggle output panel:
1. `data_scaleup/profile_baseline/profiles.json`
2. `data_scaleup/derived/labels.json`
3. `data_scaleup/manifest_static/manifest_static.json`
4. `data_scaleup_raw.zip`

Bring all four back, plus the printed **CPU model / core count / Python version** from Cell 1, and
Cell 6's gate output. Unzip `data_scaleup_raw.zip` into `data_scaleup/raw/` in the paper repo. Then in
the repo:

```
PYTHONPATH=src python -c "from cart.gates.validity import run_all_gates; from cart.selectors.registry_scaleup import SELECTORS_SCALEUP; import json; print(json.dumps(run_all_gates('data_scaleup', selectors=SELECTORS_SCALEUP), indent=2))"
```

to confirm 6/6 locally before Phase C (evaluate/analyze) begins.